# 03 - Feature Extraction

This notebook converts reconstructed ADS-B trajectories into flight-level behavioral features. It uses the same feature contracts as the automated pipeline, while avoiding an accidental full trajectory rebuild unless explicitly enabled.

## 1. Setup and Controls

In [ ]:
import importlib
import os
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import main as pipeline_main
import src.feature_engineering as feature_engineering_module
from src.data_validator import DataValidator
from src.schemas import FEATURES_SCHEMA
from src.utils import load_config

pipeline_main = importlib.reload(pipeline_main)
feature_engineering_module = importlib.reload(feature_engineering_module)
FeatureExtractor = feature_engineering_module.FeatureExtractor

config = load_config(PROJECT_ROOT / 'configs' / 'config.yaml')
paths = config['paths']
processed_dir = PROJECT_ROOT / paths['processed_data_dir']

# Usually keep this False if notebook 02 already rebuilt trajectories.
# Set True only when you want this notebook to run the full `main.py --stage features --force` path.
REBUILD_TRAJECTORIES_AND_FEATURES = False

# This is the normal notebook path: reuse the current trajectories.parquet and regenerate trajectory_features.parquet.
REBUILD_FEATURES_FROM_EXISTING_TRAJECTORIES = True

FEATURE_EXTRACTION_WORKERS = max(1, min(6, (os.cpu_count() or 2) - 1))
FEATURE_PARTITION_BATCH_SIZE = int(config.get('trajectory', {}).get('feature_partition_batch_size', 200_000))
FEATURE_WORKER_BATCH_SIZE = int(config.get('trajectory', {}).get('feature_worker_batch_size', 200_000))

traj_path = processed_dir / paths['trajectories_file']
feature_output_path = processed_dir / paths['features_file']

print(f'Project root: {PROJECT_ROOT}')
print(f'Trajectory input: {traj_path}')
print(f'Feature output: {feature_output_path}')

## 2. Inspect Trajectory Input

In [ ]:
if not traj_path.exists():
    print('Trajectories file missing. Run notebook 02 with full reconstruction enabled, or run main.py --stage features --force.')
else:
    traj_parquet = pq.ParquetFile(traj_path)
    preview_columns = [
        'trajectory_id', 'icao24', 'callsign', 'timestamp', 'latitude', 'longitude',
        'geo_altitude', 'baro_altitude', 'velocity', 'true_track', 'vertical_rate',
        'trajectory_quality_status', 'is_full_flight', 'route_coverage_fraction',
        'middle_gap_count', 'gap_fraction_of_flight',
    ]
    available_columns = [col for col in preview_columns if col in traj_parquet.schema.names]
    preview_batch = next(traj_parquet.iter_batches(batch_size=5, columns=available_columns), None)
    df_traj_preview = preview_batch.to_pandas() if preview_batch is not None else pd.DataFrame()

    print(f'Rows: {traj_parquet.metadata.num_rows:,}')
    print(f'Columns: {len(traj_parquet.schema.names):,}')
    print(f'Quality columns present: {all(c in traj_parquet.schema.names for c in ["trajectory_quality_status", "is_full_flight"])}')
    display(df_traj_preview)

## 3. Extract Flight-Level Features

In [ ]:
if REBUILD_TRAJECTORIES_AND_FEATURES:
    validator = DataValidator(mode=config.get('validation_mode', 'warn_only'), log_dir='logs')
    pipeline_main.stage_features(config, validator, force=True)
elif REBUILD_FEATURES_FROM_EXISTING_TRAJECTORIES:
    if not traj_path.exists():
        raise FileNotFoundError(f'Trajectory input not found: {traj_path}')

    extractor = FeatureExtractor(max_gap_minutes=config.get('trajectory', {}).get('max_gap_minutes', 15))
    df_features = extractor.extract_features_parallel(
        traj_path,
        num_workers=FEATURE_EXTRACTION_WORKERS,
        partition_batch_size=FEATURE_PARTITION_BATCH_SIZE,
        worker_batch_size=FEATURE_WORKER_BATCH_SIZE,
    )

    allowed_statuses = {'full_flight', 'partial_end_missing', 'partial_start_missing'}
    before_filter = len(df_features)
    if 'trajectory_quality_status' in df_features.columns:
        df_features = df_features[df_features['trajectory_quality_status'].isin(allowed_statuses)].copy()
    if 'num_points' in df_features.columns:
        df_features = df_features[df_features['num_points'] >= 10].copy()
    if 'flight_duration' in df_features.columns:
        df_features = df_features[df_features['flight_duration'].between(300, 86400)].copy()

    print(f'Feature rows before quality gates: {before_filter:,}')
    print(f'Feature rows after quality gates: {len(df_features):,}')
    if 'trajectory_quality_status' in df_features.columns:
        display(df_features['trajectory_quality_status'].value_counts(dropna=False).rename_axis('trajectory_quality_status').reset_index(name='flight_count'))

    validator = DataValidator(mode=config.get('validation_mode', 'warn_only'), log_dir='logs')
    validator.validate(
        df_features,
        pipeline_main._stage_schema(FEATURES_SCHEMA, df_features),
        'trajectory_features',
    )

    extractor.save_features(df_features, feature_output_path)
    print(f'Saved ADS-B feature set to {feature_output_path}')
    display(df_features.head())
else:
    print('Feature extraction disabled by notebook controls.')

## 4. Feature Output Summary

In [ ]:
if feature_output_path.exists():
    df_features_preview = pd.read_parquet(feature_output_path)
    print(f'Feature file rows: {len(df_features_preview):,}')
    print(f'Feature file columns: {len(df_features_preview.columns):,}')
    important_cols = [
        'trajectory_id', 'trajectory_quality_status', 'trajectory_quality_score',
        'is_full_flight', 'route_coverage_fraction', 'middle_gap_count',
        'gap_fraction_of_flight', 'max_inter_ping_seconds', 'ping_interval_cv',
    ]
    display(df_features_preview[[c for c in important_cols if c in df_features_preview.columns]].head())
else:
    print('Feature output does not exist yet.')